# Image models, Danenberg → Jackson (the reverse direction of the main figure)

**Models, scored on the Jackson test patients:**
1. **ImmuVis trained on Danenberg** (`immuvis-danenberg-train`): the mirror of the Jackson-trained ImmuVis in the main figure.
2. **EVA, zero-shot.** Its raw correlations must reproduce `cross-cohort-metrics-jackson` v1.

**Scoring:** the same cells, truth and metrics as `cross-cohort-metrics-jackson`.
- Cell order and truth sums are read from that kernel's outputs and checked against truth maps rebuilt here from the Jackson TIFFs and the `jackson-cells-prep` masks.
- **Correlations** use each model's raw output.
- **Errors (RMSE)** use the output mapped to the target space without any test data:
  - ImmuVis: 2.3 × output, i.e. the clipped arcsinh value.
  - EVA: arcsinh value = lo + output × (hi − lo), where lo / hi are the medians, over the **Danenberg train** images, of the hidden marker's per-image p0.5 / p99.5.
- **Constant-prediction baseline:** the same Danenberg train images give the training-cohort mean (`source_reference_danenberg.csv`).

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Inputs: Jackson test cells and truth (from `cross-cohort-metrics-jackson`), Danenberg reference statistics

In [ ]:
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs"], check=True)   # 2024.9.22 no longer offered
import tifffile
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()

tifs = g("*.tiff")
def _is_mask(p):
    return "/masks/" in p.replace("\\", "/")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if os.path.basename(p).startswith("MB_") and not _is_mask(p)}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if os.path.basename(p).startswith("MB_") and _is_mask(p)}
J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if not os.path.basename(p).startswith("MB_") and not _is_mask(p) and "jackson" in p.lower()}
J_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if not os.path.basename(p).startswith("MB_") and _is_mask(p)}
dix = {m: i for i, m in enumerate(D_CHAN)}
jix = {m: i for i, m in enumerate(J_CHAN)}
CACHE = os.path.dirname(g("channel_preproc_residual.pkl")[0])
_, TNS = pickle.load(open(f"{CACHE}/channel_preproc_residual.pkl", "rb"))

def r_from(n, Sg, Sp, Sgg, Spp, Sgp):
    with np.errstate(invalid="ignore", divide="ignore"):
        return (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def truth_map(im_hwc, m):
    """The image models' target space: clip(arcsinh(x/5) / p99_Jackson, 0, 1) (p01 = 0)."""
    p01, p99 = TNS[jix[m]]
    return np.clip((np.arcsinh(im_hwc[..., dix[m]] / COF) - p01) / (p99 - p01 + 1e-8), 0., 1.)

def to_target_arc(m, arc):
    """arcsinh(x/5) values -> the target space (fixed Jackson-train p01/p99 of marker m)."""
    p01, p99 = TNS[jix[m]]
    return np.clip((arc - p01) / (p99 - p01 + 1e-8), 0., 1.)

def _ref_one(img_path, mask_path, n_chan, idx):
    im = read_hwc(img_path, n_chan)
    lab = tifffile.imread(mask_path).astype(np.int64).ravel()
    assert lab.size == im.shape[0] * im.shape[1], (img_path, im.shape)
    v = lab > 0
    cnt = np.bincount(lab[v])
    ok = cnt > 0
    out = {}
    for m in MARKERS:
        arc = np.arcsinh(im[..., idx[m]] / COF)
        lo, hi = np.percentile(arc, [0.5, 99.5])
        t = to_target_arc(m, arc).ravel().astype(np.float64)
        cm = np.bincount(lab[v], t[v])[ok] / cnt[ok]
        out[m] = (lo, hi, t.sum(), t.size, cm.sum(), ok.sum())
    return out

def source_reference(names, img_map, mask_map, n_chan, idx, cohort):
    """Training-cohort statistics, per marker, that need no test data:
    eva_lo / eva_hi: median over images of the per-image arcsinh p0.5 / p99.5 (maps EVA's per-image scale back);
    pixel_mean / cell_mean: mean target-space value over all pixels / over all cells (the constant-prediction baseline)."""
    res = Parallel(n_jobs=4)(delayed(_ref_one)(img_map[s], mask_map[s], n_chan, idx) for s in names)
    rows = []
    for m in MARKERS:
        a = np.array([r[m] for r in res], np.float64)
        rows.append(dict(cohort=cohort, marker=m, n_images=len(res), eva_lo=float(np.median(a[:, 0])),
                         eva_hi=float(np.median(a[:, 1])), pixel_mean=a[:, 2].sum() / a[:, 3].sum(),
                         cell_mean=a[:, 4].sum() / a[:, 5].sum(), n_cells=int(a[:, 5].sum())))
    return pd.DataFrame(rows).set_index("marker")

def rmse_from(n, Sgg, Spp, Sgp):
    return np.sqrt(np.maximum(Sgg - 2 * Sgp + Spp, 0) / n)


# the Jackson test cells, truth cell means and truth pixel sums of cross-cohort-metrics-jackson
_npz = [p for p in g("percell_predictions_test.npz") if "jackson" in p.lower()]
assert len(_npz) == 1, _npz
Z = np.load(_npz[0], allow_pickle=False)
ZI, ZO = Z["image"], Z["object_number"].astype(np.int64)
TRUTHJ = {m: Z[f"truth__{m}"] for m in MARKERS}
_s0 = pd.read_csv(os.path.join(os.path.dirname(_npz[0]), "pixel_sums_test.csv"))
TS = _s0[_s0.model == "Table"].set_index(["marker", "image"])
TEST_NAMES = list(dict.fromkeys(ZI.tolist()))
SPAN = {}
for s in TEST_NAMES:
    ix = np.flatnonzero(ZI == s)
    assert (np.diff(ix) == 1).all(), s
    SPAN[s] = (ix[0], ix[-1] + 1)
assert all(s in J_IMG and s in J_MASK for s in TEST_NAMES)
NZ = len(ZI)
print(f"Jackson test: {NZ:,} cells in {len(TEST_NAMES)} images")

# Danenberg train images (replicate 0 of train-test-grid, the patients ImmuVis and the Danenberg cell models trained on)
D_TRVA_PAT = np.unique(DD["patient"][DD["split"] != "test"])
_uv = np.random.default_rng(SEED + 21 + 0).random(len(D_TRVA_PAT))
_dval = set(D_TRVA_PAT[_uv < 0.125])
D_SPLIT = np.where(DD["split"] == "test", "test", np.where(pd.Series(DD["patient"]).isin(_dval).values, "val", "train"))
_isp = pd.Series(D_SPLIT, index=DD["cells"].image_name.values).groupby(level=0).first()
D_TRAIN = sorted(s for s in _isp.index[_isp == "train"] if s in D_IMG and s in D_MASK)
REF = source_reference(D_TRAIN, D_IMG, D_MASK, len(D_CHAN), dix, "Danenberg")
REF.to_csv(f"{OUT}/source_reference_danenberg.csv")
print(REF.round(4).to_string())
tick(f"Danenberg reference statistics from {len(D_TRAIN)} train images")

PATCH = 32                                        # 32 µm patches (1 µm/px), non-overlapping, partial edge tiles dropped

def patch_means(A):
    h, w = (A.shape[0] // PATCH) * PATCH, (A.shape[1] // PATCH) * PATCH
    return A[:h, :w].reshape(h // PATCH, PATCH, w // PATCH, PATCH).mean((1, 3)).ravel()

def patch_row(MODEL, m, s, G2, P2, T2=None):
    """Per-image sums over 32 µm patch means (truth G2, raw prediction P2, target-scale prediction T2)."""
    g, p = patch_means(G2), patch_means(P2)
    row = dict(model=MODEL, marker=m, image=s, n=g.size, Sg=g.sum(), Sgg=g @ g, Sp=p.sum(), Spp=p @ p, Sgp=g @ p)
    if T2 is not None:
        t = patch_means(T2)
        row.update(Sp_t=t.sum(), Spp_t=t @ t, Sgp_t=g @ t)
    if len(PATCH_CHECK) < 4 and g.size > 2 and g.std() > 0 and p.std() > 0:   # sums must give the direct patch r
        PATCH_CHECK.append(abs(float(np.corrcoef(g, p)[0, 1]) - float(r_from(*(row[c] for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"])))))
    return row

PATCH_CHECK = []

def score_jackson(MODEL, predict_image, to_target):
    """predict_image(im_hwc) -> {marker: map}. Raw pixel sums + target-scale sums (_t); raw + target-scale cell means."""
    rows, chk, prow = [], [], []
    CELLP = {m: np.full(NZ, np.nan, np.float32) for m in MARKERS}
    CELLT = {m: np.full(NZ, np.nan, np.float32) for m in MARKERS}
    for j, s in enumerate(TEST_NAMES):
        a, b = SPAN[s]
        im = read_hwc(J_IMG[s], len(J_CHAN))
        mask = tifffile.imread(J_MASK[s]).astype(np.int64)
        assert mask.shape == im.shape[:2], (s, mask.shape, im.shape)
        obj = ZO[a:b]
        lut = np.full(max(int(mask.max()), int(obj.max())) + 1, -1, np.int64)
        lut[obj] = np.arange(a, b)
        loc = lut[mask].ravel()
        v = loc >= 0
        cnt = np.bincount(loc[v], minlength=NZ)
        got = cnt > 0
        preds = predict_image(im)
        for m in MARKERS:
            G = to_target_arc(m, np.arcsinh(im[..., jix[m]] / COF)).astype(np.float64).ravel()
            ts = TS.loc[(m, s)]
            assert abs(G.sum() - ts.Sg) <= 1e-6 * max(1.0, ts.Sg) and G.size == ts.n, (s, m)
            P = preds[m].astype(np.float64)
            Pf = P.ravel()
            T = to_target(m, P).astype(np.float64).ravel()
            row = dict(model=MODEL, marker=m, image=s, n=G.size, Sg=G.sum(), Sgg=G @ G, Sp=Pf.sum(), Spp=Pf @ Pf,
                       Sgp=G @ Pf, Sp_t=T.sum(), Spp_t=T @ T, Sgp_t=G @ T)
            rows.append(row)
            prow.append(patch_row(MODEL, m, s, G.reshape(P.shape), P, T.reshape(P.shape)))
            if j < 2:     # RMSE from the sums must equal the direct RMSE
                chk.append(dict(image=s, marker=m, rmse_direct=float(np.sqrt(np.mean((G - T) ** 2))),
                                rmse_sums=float(rmse_from(row["n"], row["Sgg"], row["Spp_t"], row["Sgp_t"]))))
            CELLP[m][got] = (np.bincount(loc[v], Pf[v], NZ)[got] / cnt[got]).astype(np.float32)
            CELLT[m][got] = (np.bincount(loc[v], T[v], NZ)[got] / cnt[got]).astype(np.float32)
        if j % 20 == 0:
            tick(f"{MODEL}: scored {j + 1}/{len(TEST_NAMES)} Jackson test images")
    chk = pd.DataFrame(chk)
    assert (chk.rmse_direct - chk.rmse_sums).abs().max() < 1e-6, chk
    S = pd.DataFrame(rows)
    tag = MODEL.lower().split()[0]
    S.to_csv(f"{OUT}/pixel_sums_test_{tag}.csv", index=False)
    pd.DataFrame(prow).to_csv(f"{OUT}/patch_sums_test_{tag}.csv", index=False)
    assert PATCH_CHECK and max(PATCH_CHECK) < 1e-5, PATCH_CHECK
    np.savez_compressed(f"{OUT}/percell_pred_{tag}.npz", image=ZI, object_number=ZO,
                        **{f"pred__{MODEL}__{m}": CELLP[m] for m in MARKERS},
                        **{f"predt__{MODEL}__{m}": CELLT[m] for m in MARKERS})
    out = []
    for m in MARKERS:
        d = S[S.marker == m]
        ri = r_from(d.n.values, d.Sg.values, d.Sp.values, d.Sgg.values, d.Spp.values, d.Sgp.values)
        out.append(dict(marker=m, pooled=float(r_from(*(d[c].sum() for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]))),
                        per_image=float(np.nanmean(ri)),
                        pixel_rmse=float(rmse_from(d.n.sum(), d.Sgg.sum(), d.Spp_t.sum(), d.Sgp_t.sum())),
                        per_image_rmse=float(np.mean(rmse_from(d.n.values, d.Sgg.values, d.Spp_t.values, d.Sgp_t.values))),
                        cell_rmse=float(np.sqrt(np.nanmean((CELLT[m] - TRUTHJ[m]) ** 2)))))
    out = pd.DataFrame(out)
    out.to_csv(f"{OUT}/metrics_point_{tag}.csv", index=False)
    print(out.round(3).to_string(index=False))
    print("mean over 10 markers (ERα excluded):",
          out[out.marker != "estrogen_receptor_alpha"].drop(columns="marker").mean().round(3).to_dict())
    return S

## ImmuVis trained on Danenberg

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "comet_ml", "ruamel.yaml"], check=True)
import zipfile, urllib.request, copy
from torch.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
from skimage.filters import butterworth

CODE = "/tmp/immuvis_code"
def fetch_code():
    try:
        urllib.request.urlretrieve("https://anonymous.4open.science/api/repo/ImmuVis-9173/zip", "/tmp/immuvis.zip")
        zipfile.ZipFile("/tmp/immuvis.zip").extractall(CODE)
        return "anonymous.4open.science/r/ImmuVis-9173"
    except Exception as e:
        print("anonymous release not available:", e)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dav3794/multiplex-image-model.git", CODE], check=True)
        return "github.com/dav3794/multiplex-image-model"
CODE_SOURCE = fetch_code()
root = CODE if os.path.isdir(f"{CODE}/multiplex_model") else os.path.dirname(glob.glob(f"{CODE}/**/multiplex_model", recursive=True)[0])
sys.path.insert(0, root)
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import (EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking,
                                   ClampWithGrad, get_scheduler_with_warmup)
from multiplex_model.losses import beta_nll_loss
print("ImmuVis code from", CODE_SOURCE)
# cuDNN on a T4 ran this model at ~6 s/step and leaked ~50 MB host RAM per step (immuvis-profile);
# with cuDNN off: 0.58 s/step and flat RAM
torch.backends.cudnn.enabled = False

CH = NUC + MARKERS                         # 14 shared channels = 14 marker tokens
IDS = torch.arange(len(CH))
CLIP = 2.3
CROP, BATCH = 128, 16
TRAIN_MIN = float(os.environ.get("TRAIN_MIN", 180))
EVAL_EVERY = 500

def immuvis_pre(im_hwc, names, idx):
    x = np.arcsinh(np.stack([im_hwc[..., idx[c]] for c in names]) / COF)
    x = np.stack([butterworth(c, cutoff_frequency_ratio=0.2, high_pass=False) for c in x])
    return (np.clip(x, 0, CLIP) / CLIP).astype(np.float16)

ENC = EncoderConfig(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4],
                    pm_embedding_dims=[192, 384, 768],
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
DEC = DecoderConfig(decoded_embed_dim=512, num_blocks=1,
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()

In [ ]:
CK = torch.load(g("immuvis_danenberg_best.pth")[0], map_location="cpu", weights_only=False)
assert CK["channels"] == CH and CK["clip"] == CLIP and CK["train_cohort"] == "danenberg"
net = MultiplexAutoencoder(num_channels=len(CH), encoder_config=ENC, decoder_config=DEC).to(dev)
net.load_state_dict(CK["model_state_dict"])
net.eval()
print(f"checkpoint: step {CK['step']}/{CK['total_steps']}, Danenberg val LOO pixel r {CK['best_val_loo_pixel_r']:.3f}, "
      f"{CK['n_train_images']} train images")

T_IN, M_CTX = 128, 32

@torch.no_grad()
def immuvis_predict(im_hwc):
    x = immuvis_pre(im_hwc, CH, jix)               # [14, H, W], Jackson channel order mapped by name
    C, H, W = x.shape
    tiles = [(y0, x0) for y0 in range(0, H, T_IN) for x0 in range(0, W, T_IN)]
    win = T_IN + 2 * M_CTX
    xp = np.pad(x, ((0, 0), (M_CTX, M_CTX + T_IN), (M_CTX, M_CTX + T_IN)))
    batch = torch.from_numpy(np.stack([xp[:, y0:y0 + win, x0:x0 + win] for y0, x0 in tiles]).astype(np.float32))
    preds = {}
    for k in range(3, len(CH)):
        act = [i for i in range(len(CH)) if i != k]
        outs = []
        for b0 in range(0, len(batch), 64):
            xb = batch[b0:b0 + 64].to(dev)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lat = net.encode(xb[:, act], IDS[act].to(dev).expand(len(xb), -1))["output"]
                o = net.decode(lat, IDS[[k]].to(dev).expand(len(xb), -1))
            outs.append(torch.sigmoid(o[..., 0].float())[:, 0, M_CTX:M_CTX + T_IN, M_CTX:M_CTX + T_IN].cpu().numpy())
        outs = np.concatenate(outs)
        P = np.zeros((H + T_IN, W + T_IN), np.float32)
        for (y0, x0), o in zip(tiles, outs):
            P[y0:y0 + T_IN, x0:x0 + T_IN] = o
        preds[CH[k]] = P[:H, :W]
    return preds

@guard("ImmuVis")
def _():
    # ImmuVis output is clip(filtered arcsinh, 0, 2.3) / 2.3, so CLIP * output is back in arcsinh units
    score_jackson("ImmuVis", immuvis_predict, lambda m, P: to_target_arc(m, CLIP * P))
del net
if dev == "cuda":
    torch.cuda.empty_cache()
torch.backends.cudnn.enabled = True     # cuDNN was turned off only for ImmuVis; EVA ran with it in cross-cohort-metrics-jackson

## EVA, zero-shot

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omegaconf", "einops"], check=True)
EVA_DIR = "/tmp/Eva"
if not os.path.isdir(EVA_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/YAndrewL/Eva.git", EVA_DIR], check=True)
EVA_COMMIT = subprocess.run(["git", "-C", EVA_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.makedirs(f"{EVA_DIR}/marker_embeddings", exist_ok=True)
gp = g("GenePT_embedding.pkl")[0]
if not os.path.exists(f"{EVA_DIR}/marker_embeddings/GenePT_embedding.pkl"):
    os.symlink(gp, f"{EVA_DIR}/marker_embeddings/GenePT_embedding.pkl")
os.chdir(EVA_DIR)                                # mae.py opens marker_embeddings/ relative to the working directory
sys.path.insert(0, EVA_DIR)
from omegaconf import OmegaConf
from Eva.eva import EvaMAE
from utils.constant import marker_to_gene
conf = OmegaConf.load(f"{EVA_DIR}/config.yaml")
eva = EvaMAE.from_checkpoint(g("Eva_model.ckpt")[0], conf, device=dev)
eva.eval()
os.chdir("/kaggle/working")
print("EVA commit", EVA_COMMIT, "| parameters", sum(p.numel() for p in eva.parameters()) / 1e6, "M")

EVA_NAME = {"histone_h3": "HistoneH3", "c_erb_b_2_her2": "HER2", "cd20": "CD20", "cd3": "CD3e", "cd45": "CD45",
            "cd68": "CD68", "cytokeratin_5": "Keratin5", "cytokeratin_8_18": "Keratin8/18",
            "estrogen_receptor_alpha": "ERa", "ki_67": "Ki67", "pan_cytokeratin": "PanCK", "sma": "aSMA"}
assert all(v in marker_to_gene for v in EVA_NAME.values()) and "DAPI" in marker_to_gene
CH_EVA = ["DAPI"] + [EVA_NAME[c] for c in ["histone_h3"] + MARKERS]       # 13 channels
TILE, TOK = 224, 8
NTOK = (TILE // TOK) ** 2

def eva_channels(im_hwc, idx, mode):
    """[H, W, 13] in [0, 1]: DAPI (mean of DNA1, DNA2), HistoneH3, the 11 markers; per-image, per-channel scaling."""
    raw = [0.5 * (im_hwc[..., idx["dna1"]] + im_hwc[..., idx["dna2"]])] + [im_hwc[..., idx[c]] for c in ["histone_h3"] + MARKERS]
    out = []
    for c in raw:
        c = np.arcsinh(c / COF) if mode == "arcsinh" else c
        lo, hi = np.percentile(c, [0.5, 99.5])
        out.append(np.clip((c - lo) / (hi - lo + 1e-8), 0, 1))
    return np.stack(out, -1).astype(np.float32)

@torch.no_grad()
def eva_predict_arr(x):
    """x: [H, W, 13] scaled. Returns {marker: map}, each marker predicted with its own channel hidden."""
    H, W, C = x.shape
    Hp, Wp = -(-H // TILE) * TILE, -(-W // TILE) * TILE
    xp = np.zeros((Hp, Wp, C), np.float32)
    xp[:H, :W] = x
    tiles = [(y0, x0) for y0 in range(0, Hp, TILE) for x0 in range(0, Wp, TILE)]
    preds = {}
    for k, m in enumerate(MARKERS):
        ci = 2 + k                                  # channel of the target marker
        mask = torch.zeros(C, NTOK, device=dev)
        mask[ci] = 1.0
        P = np.zeros((Hp, Wp), np.float32)
        for b0 in range(0, len(tiles), 16):
            tb = tiles[b0:b0 + 16]
            xb = np.stack([xp[y0:y0 + TILE, x0:x0 + TILE] for y0, x0 in tb])
            xb[..., ci] = 0.0
            # EVA permutes [B, H, W, C] -> [B, C, H, W] and then calls .view, so the memory must be [B, C, H, W]
            xb = torch.from_numpy(np.ascontiguousarray(xb.transpose(0, 3, 1, 2))).to(dev).permute(0, 2, 3, 1)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                rec, _, _ = eva(xb, marker_in=[CH_EVA] * len(tb), marker_out=[[CH_EVA[ci]]] * len(tb),
                                infer_mask=mask, channel_mask=None)
            rec = rec.float()[..., 0].cpu().numpy()
            for (y0, x0), r in zip(tb, rec):
                P[y0:y0 + TILE, x0:x0 + TILE] = r
        preds[m] = P[:H, :W]
    return preds

from torch.amp import autocast

In [ ]:
MODE = "arcsinh"            # chosen on Jackson val images in eva-zeroshot (0.163 vs 0.159 for linear)

def eva_to_target(m, P):
    lo, hi = REF.loc[m, "eva_lo"], REF.loc[m, "eva_hi"]
    return to_target_arc(m, lo + P * (hi - lo))

@guard("EVA")
def _():
    score_jackson("EVA", lambda im: eva_predict_arr(eva_channels(im, jix, MODE)), eva_to_target)

json.dump({"failed_blocks": FAILED, "minutes": (time.time() - T0) / 60, "n_test_cells": int(NZ),
           "n_test_images": len(TEST_NAMES), "n_danenberg_reference_images": len(D_TRAIN),
           "immuvis_checkpoint": {k: CK[k] for k in ["step", "total_steps", "best_val_loo_pixel_r", "train_minutes",
                                                     "n_train_images", "n_val_images", "code_source"]},
           "eva_commit": EVA_COMMIT}, open(f"{OUT}/run_info.json", "w"))
print("failed blocks:", FAILED if FAILED else "none")